# colony3

> Phase 18, Stage 3 (`specs/multiagent/phase18/stage3.md`): adaptive liars and unassigned guards at N = 16. Claude agents told to deceive in the liar slots (`fault = 'adaptive'`), and the guard role described to every honest agent with no assignment (`colonyVolunteer`), paired with Stage 1's cells on the same seeds. The grids, the pilot's compliance gate and cost rule, liar-behaviour and volunteering measures, the cell runner, the cached replay of the reused Stage 1 cells, `COLONY3_PLAN`, `stage3Summaries` and `writeStage3`.

In [ ]:
#| default_exp colony3

In [ ]:
#| hide
from nbdev.showdoc import *

## Cells, plan and gates

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm
from explRL.colony_grid import COLONY_ARMS, COLONY_SEEDS, PILOT_SEED, colonyCell
from explRL.colony_report import NAN, bootSeeds, cellVals, pairs, signFlipExact
from explRL.colony_trace import colonyColumns, replay, specIndex
from explRL.fix_grid import fixEstimate
from explRL.llm_runs import LLMConfig, cellRow, runLLMCell

ARMS3 = {**COLONY_ARMS, 'colonyVolunteer': dict(colony=True, volunteer=True)}     # COLONY_ARMS itself is frozen in COLONY_PLAN
N3, STAGE3_NEW = 16, [('plain', 'adaptive'), ('colonyGuards', 'adaptive'), ('colonyVolunteer', 'independent'),
                       ('colonyVolunteer', 'clean'), ('colonyVolunteer', 'adaptive')]
STAGE3_REUSED = [('plain', 'clean'), ('plain', 'independent'), ('colony', 'independent'), ('colony', 'clean'),
                 ('colonyGuards', 'independent'), ('colonyGuards', 'clean')]


def stage3Grid(seeds=COLONY_SEEDS):
    "Study colony3: the new Stage 3 cells at N 16 (40 cells)."
    return [colonyCell('colony3', a, N3, s, f, arms=ARMS3) for a, f in STAGE3_NEW for s in seeds]


stage3PilotGrid = lambda: [colonyCell('colony3Pilot', a, N3, PILOT_SEED, f, arms=ARMS3)     # the pilot (not analysed)
                           for a, f in (('plain', 'adaptive'), ('colonyVolunteer', 'independent'))]


def reusedCells(seeds=COLONY_SEEDS):
    "The six Stage 1 cells Stage 3 reuses (study colony1, exactly as Stage 1 ran them)."
    return [colonyCell('colony1', a, N3, s, f) for a, f in STAGE3_REUSED for s in seeds]


COLONY3_PLAN = dict(
    version=1, frozen=True, N=N3, seeds=COLONY_SEEDS, pilotSeed=PILOT_SEED, cells=[f'{a}|{f}' for a, f in STAGE3_NEW],
    reused=[f'{a}|{f}' for a, f in STAGE3_REUSED], reuse='Stage 1 cells replayed from the response cache with the Stage 3 '
    'columns added ($0); their Stage 1 columns must be unchanged', arms={a: ARMS3[a] for a in ('plain', 'colony', 'colonyGuards', 'colonyVolunteer')},
    hypotheses=dict(H18_10='colonyGuards|adaptive - plain|adaptive, honestPooled, paired on seed: one-sided exact sign-flip > 0',
                    H18_11='colonyVolunteer|independent - colony|independent (Stage 1), honestPooled, paired on seed: as H18_10'),
    holm=['H18_10', 'H18_11'], alpha=0.05,
    secondary='adaptive minus scripted honest loss (plain); share of each liar kind\'s damage guards recover; damage per lie; '
              'volunteers minus assigned guards; volunteers per board; specialization minus its null; the volunteer clean tax; '
              'liar behaviour (false posts per turn, targeting guarded clues, collusion, refusals)',
    gate=dict(minFalsePerTurn=0.10, maxRefusals=0.20, maxInvalid=0.05), costRule=dict(run=30.0, otherwise='stop and ask'),
    cap=200.0, nboot=2000, exploratory=True)


stage3Estimate = lambda cells: fixEstimate(cells, stage3PilotGrid())   # adaptive cells count all N agents; the pilot is cached


def costRule3(perCall, plan=COLONY3_PLAN):
    "Run if Stage 3 projects at most plan['costRule']['run'] dollars, else stop."
    proj = stage3Estimate(stage3Grid())['newCalls'] * perCall
    return dict(verdict='run' if proj <= plan['costRule']['run'] else 'stop', projected=proj)


def complianceGate(rows, plan=COLONY3_PLAN):
    "The pilot's gate: adaptive liars lie often enough and rarely refuse; invalid replies stay under 5%."
    g, adv = plan['gate'], [r for r in rows if str(r['level']).endswith('|adaptive')]
    fpt = float(np.mean([float(r['liarFalsePerTurn']) for r in adv])) if adv else NAN
    ref = float(np.mean([float(r['liarRefusals']) for r in adv])) if adv else NAN
    inv = sum(float(r['invalid']) for r in rows) / max(1.0, sum(float(r['calls']) for r in rows))
    reasons = [m for m, bad in ((f'liars lie in {fpt:.2f} of their turns', not fpt >= g['minFalsePerTurn']),
                                (f'liars refuse {ref:.2f} of replies', not ref <= g['maxRefusals']),
                                (f'invalid replies {inv:.3f}', not inv < g['maxInvalid'])) if bad]
    return dict(ok=not reasons, reasons=reasons, liarFalsePerTurn=fpt, liarRefusals=ref, invalid=inv)

## Measures and the cell runner

In [ ]:
#| export
def liarBehaviour(board, log, nLiar, guards=0):
    "False posts by the liar slots, per liar turn, on clues an assigned guard holds, repeating another liar's lie; refusals."
    turns = sum(1 for r in board.ledger if r[0] < nLiar)
    lies = [s for s in board.shareLog if s[1] < nLiar and s[6]]
    gs = set(range(nLiar, nLiar + guards))
    posted, again = {}, 0                                                   # again: a lie another liar posted first
    for s in lies:
        key = (s[2], s[3], s[4])
        again += any(a != s[1] for a in posted.get(key, ()))
        posted.setdefault(key, set()).add(s[1])
    said = [x for x in log if x.get('agent', -1) < nLiar and x.get('kind') == 'turn']
    return dict(liarFalsePosts=len(lies), liarFalsePerTurn=len(lies) / turns if turns else NAN,
                liarTargetGuarded=float(np.mean([any(h in gs for h in board.tasks[s[2]].holders[s[3]]) for s in lies])) if gs and lies else NAN,
                liarCollude=again / len(lies) if lies else NAN,
                liarRefusals=float(np.mean([x.get('refusal') is not None for x in said])) if said else NAN)


def volunteering(board, nLiar):
    "Behavioural guards: honest agents who open no subtask after their fifth turn and spend at least half of their turns sharing."
    rows = {i: [r for r in board.ledger if r[0] == i] for i in range(nLiar, board.nAgents)}
    shares = {i: sum(1 for s in board.shareLog if s[1] == i) for i in rows}
    return dict(volunteers=sum(bool(r) and not any(x[3] == 'travel' for x in r[5:]) and shares[i] >= len(r) / 2 for i, r in rows.items()))


def stage3Columns(cfg, rp, seed):
    "Phase 18's columns of a replay plus liar behaviour, volunteering and the specialization index of the honest slots."
    slots = [[x for x in rp.log if x.get('agent') == i] for i in range(cfg.N)]
    si, sn = specIndex(slots, cfg.nLiar, seed)
    return {**colonyColumns(rp), **liarBehaviour(rp.board, rp.log, cfg.nLiar, cfg.world.get('guards', 0)),
            **volunteering(rp.board, cfg.nLiar), 'specIndex': si, 'specNull': sn}


def runStage3(cfg, seed, client):
    "runClue's signature for runLLMCell: the replay's rows with Stage 3's columns, and its turn log."
    rp = replay(cfg, seed, client)
    return [stage3Columns(cfg, rp, seed)], rp.log


def runStage3Cell(cell):
    "For runWorkload (picklable, so worker processes can run it): a Stage 3 cell with Stage 3's columns."
    return runLLMCell(cell, run=runStage3)


def _same(a, b):
    try:
        return (a := float(a)) == (b := float(b)) or (a != a and b != b) or abs(a - b) <= 1e-9 * max(1.0, abs(a))
    except (TypeError, ValueError):
        return str(a) == str(b)


def reusedRows(client, rows1, seeds=COLONY_SEEDS):
    """The reused Stage 1 cells replayed through `client` (the cache-only client: $0) with Stage 3's columns; every column of
    the matching Stage 1 row must be unchanged."""
    out = []
    for c in reusedCells(seeds):
        cfg = LLMConfig.fromDict(c['cfg'])
        row = cellRow(c, cfg, stage3Columns(cfg, replay(cfg, c['seed'], client(cfg)), c['seed']))
        ref = next(r for r in rows1 if r['level'] == c['level'] and int(r['N']) == c['N'] and int(r['seed']) == c['seed'])
        bad = [k for k in ref if k in row and not k.startswith('world.') and not _same(ref[k], row[k])]   # the cells' configs are tested equal
        assert not bad, f"replay of {c['level']} seed {c['seed']} differs from Stage 1 in {bad[:5]}"
        out.append(row)
    return out

## Analysis

In [ ]:
#| export
def _test(dp, nboot):
    est, lo, hi = bootSeeds(dp, nboot=nboot)
    return dict(n=len(dp), estimate=est, ci=[lo, hi], p=signFlipExact(list(dp.values())))


def _loss(rows, arm, fault):
    "{seed: welfareHonest clean - with the fault}, same arm."
    c, f = cellVals(rows, arm, 'clean', 'welfareHonest', [N3]), cellVals(rows, arm, fault, 'welfareHonest', [N3])
    return {s: c[(s, N)] - v for (s, N), v in f.items() if (s, N) in c}


def stage3Summaries(rows3, rowsReused, plan=COLONY3_PLAN, nboot=2000):
    "H18-10 and H18-11 (Holm), and the secondary measures, from the new rows and the replayed Stage 1 references."
    rows = list(rows3) + list(rowsReused)
    hyp = dict(H18_10=_test(pairs(rows, 'colonyGuards', 'honestPooled', 'adaptive', 'plain', [N3]), nboot),
               H18_11=_test(pairs(rows, 'colonyVolunteer', 'honestPooled', 'independent', 'colony', [N3]), nboot))
    for h, q in zip(plan['holm'], holm([hyp[h]['p'] for h in plan['holm']])):
        hyp[h].update(pHolm=float(q), supported=bool(q < plan['alpha'] and hyp[h]['estimate'] > 0))
    la, ls = _loss(rows, 'plain', 'adaptive'), _loss(rows, 'plain', 'independent')
    rec = lambda f: float(1 - np.mean(list(_loss(rows, 'colonyGuards', f).values())) / np.mean(list(_loss(rows, 'plain', f).values())))
    lies = lambda a, f: cellVals(rows, a, f, 'liarFalsePosts', [N3])
    perLie = {f'{a}|{f}': float(np.mean([v / lies(a, f)[(s, N3)] for s, v in _loss(rows, a, f).items() if lies(a, f).get((s, N3))]))
              for a in ('plain', 'colonyGuards') for f in ('independent', 'adaptive')}
    mean = lambda a, f, k: float(np.nanmean(list(cellVals(rows, a, f, k, [N3]).values()) or [NAN]))
    cells = sorted({str(r['level']) for r in rows})
    keys = ('honestPooled', 'contamination', 'volunteers', 'specIndex', 'specNull', 'liarFalsePerTurn', 'liarTargetGuarded', 'liarCollude', 'liarRefusals')
    sec = dict(adaptiveMinusScripted=dict(zip(('estimate', 'lo', 'hi'), bootSeeds({(s, N3): la[s] - ls[s] for s in la if s in ls}, nboot=nboot))),
               recovered=dict(independent=rec('independent'), adaptive=rec('adaptive')), damagePerLie=perLie,
               volunteersMinusGuards=_test(pairs(rows, 'colonyVolunteer', 'honestPooled', 'independent', 'colonyGuards', [N3]), nboot),
               volunteerSpec=_test({k: v - cellVals(rows, 'colonyVolunteer', 'independent', 'specNull', [N3])[k]            # index minus
                                    for k, v in cellVals(rows, 'colonyVolunteer', 'independent', 'specIndex', [N3]).items()}, nboot),   # its null
               volunteerTax=dict(zip(('estimate', 'lo', 'hi'), bootSeeds({k: -v for k, v in pairs(rows, 'colonyVolunteer', 'pooledShare', 'clean', 'plain', [N3]).items()}, nboot=nboot))),
               cells={c: {k: mean(*c.split('|'), k) for k in keys} for c in cells})
    return dict(hyp=hyp, secondary=sec)


_f = lambda x: 'n/a' if x is None or x != x else f'{x:.3f}'


def writeStage3(path, S, meta):
    "report.md for Stage 3."
    L = ['# Phase 18, Stage 3: adaptive liars and unassigned guards (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| hypothesis | n | estimate | 95% interval | p (one-sided, exact) | Holm | supported |', '|---|---|---|---|---|---|---|']
    for h, t in S['hyp'].items():
        L.append(f"| {h} | {t['n']} | {_f(t['estimate'])} | [{_f(t['ci'][0])}, {_f(t['ci'][1])}] | {_f(t['p'])} | {_f(t['pHolm'])} | {t['supported']} |")
    s = S['secondary']
    a, v, x = s['adaptiveMinusScripted'], s['volunteersMinusGuards'], s['volunteerTax']
    L += ['', f"Adaptive minus scripted honest loss (plain board): {_f(a['estimate'])} [{_f(a['lo'])}, {_f(a['hi'])}]",
          f"Share of the damage guards recover: scripted {_f(s['recovered']['independent'])}, adaptive {_f(s['recovered']['adaptive'])}",
          'Damage per lie: ' + ', '.join(f'{k} {_f(w)}' for k, w in s['damagePerLie'].items()),
          f"Volunteers minus assigned guards (scripted liars): {_f(v['estimate'])} [{_f(v['ci'][0])}, {_f(v['ci'][1])}], p {_f(v['p'])}",
          f"Volunteer clean tax (plain minus volunteers, pooled share): {_f(x['estimate'])} [{_f(x['lo'])}, {_f(x['hi'])}]",
          f"Volunteers' specialization minus its shuffled null: {_f(s['volunteerSpec']['estimate'])}, p {_f(s['volunteerSpec']['p'])}", '']
    ks = list(next(iter(s['cells'].values())))
    L += ['| cell | ' + ' | '.join(ks) + ' |', '|---|' + '---|' * len(ks)]
    L += [f'| {c} | ' + ' | '.join(_f(d[k]) for k in ks) + ' |' for c, d in s['cells'].items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## Tests

In [ ]:
import json, tempfile, os
from types import SimpleNamespace
from explRL.colony3 import *
from explRL.colony_grid import syntheticRows18
from explRL.coop_grid import coopHash
from explRL.llm_runs import LLMConfig
from explRL.clue_text import TextClueEnv
from explRL.clueboard import ClueBoardEnv

# grids, estimate (adaptive cells count all 16 agents), cost rule
g = stage3Grid()
assert len(g) == 40 and len(stage3PilotGrid()) == 2 and len(reusedCells()) == 48
for c in g:
    cfg = LLMConfig.fromDict(c['cfg'])
    arm, fault = c['level'].split('|')
    assert cfg.N == 16 and cfg.nLiar == (0 if fault == 'clean' else 4) and cfg.world.get('volunteer', False) == (arm == 'colonyVolunteer')
    assert cfg.world.get('fault', 'independent') == fault or fault in ('clean', 'independent')
assert stage3Estimate(g)['newCalls'] == 8 * (3 * 640 + 640 + 480)
assert costRule3(30 / 24320)['verdict'] == 'run' and costRule3(31 / 24320)['verdict'] == 'stop'
# reused cells are exactly Stage 1's
from explRL.colony_grid import stage1Grid
s1 = {(c['level'], c['N'], c['seed']): c['cfg'] for c in stage1Grid()}
assert all(s1[(c['level'], c['N'], c['seed'])] == c['cfg'] for c in reusedCells())
# the share exemption: only an adaptive liar slot may post a residue it does not know
b = ClueBoardEnv(0, 8, m=2, r=2, R=30)
k = b.open[0]; p, r = b.tasks[k].task.clue(0)
fake = (r + 1) % p
for fault, i, ok in (('adaptive', 0, True), ('adaptive', 3, False), ('independent', 0, False)):
    env = TextClueEnv(b, nLiar=2, fault=fault)
    env._known = {i: {k: {}}}
    act, _, valid = env.parse(dict(action='share', target=f'T{k}', clue=f'x mod {p} = {fake}'), i)
    assert valid == ok and (act == ('share', (k, 0, fake))) == ok, (fault, i)
# liar behaviour and volunteering on a hand-made board
bd = SimpleNamespace(nAgents=6, tasks={7: SimpleNamespace(holders=[frozenset({2, 4}), frozenset({1, 5})])},
                     shareLog=[(0.0, 0, 7, 0, 1, False, True), (1.0, 1, 7, 0, 1, True, True), (2.0, 1, 7, 1, 0, False, True),
                               (3.0, 2, 7, 1, 2, False, False)] + [(t, 3, 7, 0, 0, False, False) for t in range(4, 10)],
                     ledger=[(0, 0.0, 1, 'stay', 7)] * 4 + [(1, 0.0, 1, 'stay', 7)] * 2 + [(3, float(t), 1, 'stay', 7) for t in range(8)]
                            + [(4, 0.0, 1, 'travel', 7)] * 7)
lb = liarBehaviour(bd, [dict(agent=0, kind='turn', refusal=None), dict(agent=1, kind='turn', refusal='x')], nLiar=2, guards=1)
assert lb['liarFalsePosts'] == 3 and lb['liarFalsePerTurn'] == 0.5 and abs(lb['liarTargetGuarded'] - 2 / 3) < 1e-12
assert abs(lb['liarCollude'] - 1 / 3) < 1e-12 and lb['liarRefusals'] == 0.5
assert volunteering(bd, 2)['volunteers'] == 1                       # agent 3: 8 turns, no open, 6 shares; agent 4 opens late
# the compliance gate both ways
row = lambda lv, fpt, ref, inv=0: dict(level=lv, liarFalsePerTurn=fpt, liarRefusals=ref, invalid=inv, calls=100)
assert complianceGate([row('plain|adaptive', 0.3, 0.0), row('colonyVolunteer|independent', NAN, NAN)])['ok']
assert not complianceGate([row('plain|adaptive', 0.05, 0.0)])['ok'] and not complianceGate([row('plain|adaptive', 0.3, 0.5)])['ok']
assert not complianceGate([row('plain|adaptive', 0.3, 0.0, inv=6)])['ok']
# the analysis on synthetic rows: guards +0.1 against adaptive liars, volunteers +0.05 over the colony board
base = syntheticRows18({'plain': (0, 0, 0, 0), 'colony': (0.05, 0, 0, 0), 'colonyGuards': (0.15, 0, 0, 0), 'colonyVolunteer': (0.10, -0.02, 0, 0)},
                       sizes=(16,), noise=0.0)
rows = [dict(r, liarFalsePosts=20, volunteers=2, specIndex=2.0, specNull=1.0, liarFalsePerTurn=0.4, liarTargetGuarded=NAN, liarCollude=0.1, liarRefusals=0.0)
        for r in base]
rows += [dict(r, level=r['level'].replace('|independent', '|adaptive'), honestPooled=float(r['honestPooled']) - 0.05,
              welfareHonest=float(r['welfareHonest']) - 0.05) for r in rows if r['level'].endswith('|independent')]
reused = [r for r in rows if tuple(r['level'].split('|')) in STAGE3_REUSED]
S = stage3Summaries([r for r in rows if r not in reused], reused, nboot=100)
assert abs(S['hyp']['H18_10']['estimate'] - 0.15) < 1e-9 and S['hyp']['H18_10']['supported'] and S['hyp']['H18_10']['p'] == 1 / 256
assert abs(S['hyp']['H18_11']['estimate'] - 0.05) < 1e-9 and S['hyp']['H18_11']['pHolm'] == 2 / 256
assert abs(S['secondary']['adaptiveMinusScripted']['estimate'] - 0.05) < 1e-9 and abs(S['secondary']['volunteerTax']['estimate'] - 0.02) < 1e-9
assert abs(S['secondary']['volunteerSpec']['estimate'] - 1.0) < 1e-9 and S['secondary']['volunteerSpec']['p'] == 1 / 256
with tempfile.TemporaryDirectory() as d:
    writeStage3(f'{d}/r.md', S, dict(mode='test'))
    assert 'H18_10' in open(f'{d}/r.md').read()
json.dumps(COLONY3_PLAN); assert coopHash(COLONY3_PLAN)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()